# Tutorial: ACE Coding Assistant Training Workflow

Audience: ACE contributors using a Windows CPU laptop for inference and Colab or Kaggle for optional training.

Prerequisites: Python 3.11 or 3.12, basic command-line use, and access to this repository from the selected runtime. GPU training is optional; the native baseline is small but slow on CPU.

Learning goals:
- Prepare a documented, license-checked code corpus before training.
- Train and resume the native ACE and separate Qwen LoRA tracks.
- Save run artifacts and evaluate both tracks on the same compile/test suite.

This notebook is a command guide. Run the steps you need; training cells can take a long time.

## Outline

1. Connect the notebook to the ACE repository in the active kernel.
2. Install and prepare train/validation data.
3. Train native ACE and Qwen2.5-Coder LoRA separately.
4. Resume checkpoints, export artifacts, and evaluate on a machine with Docker.

The Qwen base weights and tokenizer remain in the pretrained track. Native ACE token IDs and bundles stay separate.

## Step 1 - Put ACE in the remote runtime

The Colab VS Code extension connects VS Code to a **remote Colab kernel**. It does not copy the files from your laptop into that VM. Clone your ACE repository in the remote terminal/notebook, or upload the repository there, then open this notebook from that runtime with the ACE repository root as the working directory. See the [Colab extension quick start](https://marketplace.visualstudio.com/items?itemName=google.colab).

Kaggle notebooks also run in a remote environment. Enable Internet if you intend to clone public sources or download the model. `/kaggle/input` is read-only; write training data and checkpoints under `/kaggle/working`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path(os.environ.get("ACE_REPO_DIR", Path.cwd())).resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(
        "ACE is not in this kernel. Clone or upload the repository into the remote runtime, "
        "then set ACE_REPO_DIR to its root."
    )
print(f"ACE root: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

## Step 2 - Install the command line tools

On the Windows laptop, use the commands in `docs/TRAINING_GUIDE.md`. In Colab or Kaggle, install into the selected remote kernel. The `finetune` extra provides Transformers and PEFT; CUDA quantization tools are separate and are not needed for this workflow.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[finetune]"],
    cwd=ROOT,
    check=True,
)

## Step 3 - Prepare the repository corpus first

The source manifest selects permissively licensed projects and a train/validation repository split for all seven languages. Preparation resolves each checkout to an exact commit, verifies the declared license file, records file hashes and provenance, and writes a training-only tokenizer. Keep the resulting manifest and `licenses/` directory with the run.

Review `configs/code_sources.json` before enabling network access. Use a new output directory for a fresh corpus.

In [ ]:
CORPUS = ROOT / "data" / "coding-corpus"
if not CORPUS.exists():
    subprocess.run(
        [
            sys.executable, "-m", "scripts.prepare_repositories",
            "--sources", "configs/code_sources.json",
            "--output", str(CORPUS),
            "--config", "configs/ace_laptop.yaml",
        ],
        cwd=ROOT,
        check=True,
    )
print((CORPUS / "manifest.json").read_text(encoding="utf-8")[:1200])

## Step 4 - Build supervised instruction and FIM records

The supplied instruction examples are ACE-authored under the repository license and use separate project IDs for train and validation. Replace or extend them only with documented sources and project-level split isolation. The prepared JSONL includes both instruction examples and deterministic FIM examples from the repository corpus.

In [ ]:
SFT_DATA = ROOT / "data" / "coding-sft"
if not SFT_DATA.exists():
    subprocess.run(
        [
            sys.executable, "-m", "scripts.prepare_sft_data",
            "--code-data", str(CORPUS),
            "--instruction-data", "examples/coding_instructions.jsonl",
            "--output", str(SFT_DATA),
            "--seed", "42",
        ],
        cwd=ROOT,
        check=True,
    )
print((SFT_DATA / "manifest.json").read_text(encoding="utf-8"))

## Step 5 - Train native ACE from scratch

This first trains the native architecture on the prepared token streams. Adjust the step budget and context to the GPU/runtime you selected. It saves resumable checkpoints every 25 steps and a portable bundle at the last completed step. The command's printed bundle path is the actual path to use below.

In [ ]:
NATIVE_PRETRAIN = ROOT / "artifacts" / "native-pretrain"
subprocess.run(
    [
        sys.executable, "-m", "scripts.pretrain",
        "--data", str(CORPUS),
        "--output", str(NATIVE_PRETRAIN),
        "--max-steps", "200",
        "--max-seconds", "900",
        "--warmup-steps", "10",
        "--seq-len", "256",
        "--batch-size", "1",
        "--gradient-accumulation-steps", "4",
        "--checkpoint-every", "25",
        "--eval-every", "50",
        "--device", "cuda",
    ],
    cwd=ROOT,
    check=True,
)

### Resume native pretraining after interruption

Keep the same output directory, corpus, context, batch size, seed, learning rate, and total `--max-steps`. `--max-seconds` can change between sessions. Copy the entire run directory, including `checkpoints/` and `run.json`, before moving to another platform.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "scripts.pretrain",
        "--data", str(CORPUS),
        "--output", str(NATIVE_PRETRAIN),
        "--max-steps", "200",
        "--max-seconds", "900",
        "--warmup-steps", "10",
        "--seq-len", "256",
        "--batch-size", "1",
        "--gradient-accumulation-steps", "4",
        "--checkpoint-every", "25",
        "--eval-every", "50",
        "--device", "cuda",
        "--resume",
    ],
    cwd=ROOT,
    check=True,
)

## Step 6 - Instruction-tune native ACE

Set `BASE_BUNDLE` to the `bundle-...` path reported by native pretraining. This produces a new native ACE bundle and manifest. If the SFT run is interrupted, rerun this command with `--resume` and the same schedule.

In [ ]:
BASE_BUNDLE = NATIVE_PRETRAIN / "bundle-00000200"  # Replace with the printed bundle path if fewer steps completed.
NATIVE_SFT = ROOT / "artifacts" / "native-sft"
subprocess.run(
    [
        sys.executable, "-m", "scripts.sft",
        "--data", str(SFT_DATA),
        "--init-from", str(BASE_BUNDLE),
        "--output", str(NATIVE_SFT),
        "--max-steps", "200",
        "--max-seconds", "900",
        "--warmup-steps", "10",
        "--seq-len", "256",
        "--batch-size", "1",
        "--gradient-accumulation-steps", "4",
        "--checkpoint-every", "25",
        "--eval-every", "50",
        "--device", "cuda",
    ],
    cwd=ROOT,
    check=True,
)

## Step 7 - Fine-tune the separate pretrained track

This uses Qwen2.5-Coder-0.5B-Instruct's own tokenizer and FIM markers. The training run resolves and records an exact model revision. It stores only the LoRA adapter and tokenizer; base weights are loaded separately. Model card details: [Qwen2.5-Coder-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct).

In [ ]:
QWEN_RUN = ROOT / "artifacts" / "qwen-coder-lora"
subprocess.run(
    [
        sys.executable, "-m", "scripts.finetune_pretrained",
        "--data", str(SFT_DATA),
        "--output", str(QWEN_RUN),
        "--model-id", "Qwen/Qwen2.5-Coder-0.5B-Instruct",
        "--revision", "main",
        "--max-steps", "200",
        "--max-seconds", "900",
        "--warmup-steps", "10",
        "--seq-len", "1024",
        "--batch-size", "1",
        "--gradient-accumulation-steps", "8",
        "--checkpoint-every", "25",
        "--eval-every", "50",
        "--device", "cuda",
    ],
    cwd=ROOT,
    check=True,
)

### Resume the Qwen adapter run

Copy the complete `qwen-coder-lora/` directory, including its saved tokenizer, `run.json`, and `checkpoints/`. Use the same model ID/revision, data, sequence length, batch size, and schedule, then add `--resume`. If Kaggle or Colab stops the session, restart the runtime, restore the directory, and resume from the last complete checkpoint.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "scripts.finetune_pretrained",
        "--data", str(SFT_DATA),
        "--output", str(QWEN_RUN),
        "--model-id", "Qwen/Qwen2.5-Coder-0.5B-Instruct",
        "--revision", "main",
        "--max-steps", "200",
        "--max-seconds", "900",
        "--warmup-steps", "10",
        "--seq-len", "1024",
        "--batch-size", "1",
        "--gradient-accumulation-steps", "8",
        "--checkpoint-every", "25",
        "--eval-every", "50",
        "--device", "cuda",
        "--resume",
    ],
    cwd=ROOT,
    check=True,
)

## Step 8 - Evaluate and measure on the laptop

The starter suite contains 10 completion and 10 bug-fix tasks for each of Python, JavaScript, TypeScript, Java, Rust, C, and C++. Its task fixtures compile or run in Docker with network disabled, 2 CPUs, 4 GB memory, and a 5-minute limit by default. Colab/Kaggle runtimes may not expose a Docker daemon; if so, download the run artifacts and perform this evaluation on a machine where Docker is available. The report includes per-language pass rates and loss, model/tokenizer/data fingerprints, peak process memory, and measured generation speed.

In [ ]:
EVAL_SUITE = ROOT / "evaluation" / "coding_tasks"
if not EVAL_SUITE.exists():
    subprocess.run(
        [sys.executable, "-m", "scripts.create_evaluation_suite", "--output", str(EVAL_SUITE)],
        cwd=ROOT,
        check=True,
    )
print(f"Evaluation suite: {EVAL_SUITE}")
print("Run ace-evaluate-coding from a host with Docker after exporting a native bundle or adapter.")

## Step 9 - Save run outputs and resume safely

### Kaggle

Write prepared data and checkpoints under `/kaggle/working/ace-run/`. Before a session ends, make sure the last checkpoint directory is complete. Use **Save Version → Quick Save** for a snapshot of the current notebook and outputs; use **Save & Run All** only when the notebook's full run fits the session window. Enable saving notebook outputs in the version settings. After the version completes, find its output in the notebook viewer or add that notebook output as an input to a new notebook, copy the run directory from read-only `/kaggle/input` into `/kaggle/working`, then rerun the CLI with `--resume`. Kaggle describes versions as snapshots of notebook code and output files in its [Notebook guide](https://www.kaggle.com/docs/notebooks).

### Colab

Copy checkpoints to Google Drive or download an archive before disconnecting the runtime. Reconnect, clone/upload ACE again if needed, restore the run directory, and rerun the same command with `--resume`.

### Paid compute later

Choose any provider and instance manually. Set a spending limit before starting, confirm the GPU, memory, storage, and session limits, then run these same commands. This repository does not create cloud instances or start billing.

In [ ]:
# Package an artifact directory for manual download or upload to persistent storage.
import shutil

artifact_root = ROOT / "artifacts"
if artifact_root.is_dir():
    archive = shutil.make_archive(str(ROOT / "ace-artifacts"), "zip", root_dir=artifact_root)
    print(f"Created {archive}")
else:
    print("No artifacts to export yet.")

## Exercise and common pitfalls

1. Add a separately licensed instruction record for another language and verify its project ID appears in only one split.
2. Compare native ACE and Qwen adapter pass rates on the same suite. Record the exact model and tokenizer revisions with the report.
3. Add a realistic project-level task before drawing conclusions from this starter suite.

Do not point the pretrained run at a native ACE bundle, or overwrite an existing tokenizer. Keep each run directory intact when resuming. A low token loss alone does not show that generated patches pass tests.